In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

df = pd.read_csv("../Datos/Preprocesados/tcga_preprocesado.csv")
y = df['t']

columnas_a_probar = ['text_basico', 'text_lema', 'text_pos', 'text_stem_nltk', 'text_lema_nltk']
resultados_tfidf = []


for col in columnas_a_probar:
    X = df[col]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    
    vectorizador_tfidf = TfidfVectorizer(max_features=5000)
    X_train_tfidf = vectorizador_tfidf.fit_transform(X_train)
    X_test_tfidf = vectorizador_tfidf.transform(X_test)
    
    modelo_lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
    modelo_lr.fit(X_train_tfidf, y_train)
    score_lr = f1_score(y_test, modelo_lr.predict(X_test_tfidf), average='macro')
    
    modelo_nb = MultinomialNB()
    modelo_nb.fit(X_train_tfidf, y_train)
    score_nb = f1_score(y_test, modelo_nb.predict(X_test_tfidf), average='macro')
    
    modelo_svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=5000)
    modelo_svm.fit(X_train_tfidf, y_train)
    score_svm = f1_score(y_test, modelo_svm.predict(X_test_tfidf), average='macro')
    
    resultados_tfidf.append({
        'Preprocesamiento': col, 
        'Macro-F1 (Reg. Logística)': score_lr,
        'Macro-F1 (Naive Bayes)': score_nb,
        'Macro-F1 (SVM)': score_svm
    })

df_resultados_tfidf = pd.DataFrame(resultados_tfidf).sort_values(by='Macro-F1 (Reg. Logística)', ascending=False)

print(df_resultados_tfidf.to_string(index=False))

Preprocesamiento  Macro-F1 (Reg. Logística)  Macro-F1 (Naive Bayes)  Macro-F1 (SVM)
  text_lema_nltk                   0.579998                0.463813        0.603238
     text_basico                   0.578737                0.465567        0.597422
  text_stem_nltk                   0.578697                0.456440        0.608222
       text_lema                   0.578437                0.463362        0.599237
        text_pos                   0.576052                0.452919        0.587571


Realizamos lo mismo que anteriormente pero esta vez realizamos un pesado binario y comparamos los resultados para obtener:

-Cual es el mejor preprocesamieneto para este caso

-Cual es la mejor representacion

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

y = df['t']

resultados = []

for col in columnas_a_probar:
    X = df[col]
    
    # Split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
    
    vectorizador_bin = CountVectorizer(max_features=5000, binary=True)
    X_train_bin = vectorizador_bin.fit_transform(X_train)
    X_test_bin = vectorizador_bin.transform(X_test)
    
    modelo_lr = LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42)
    modelo_lr.fit(X_train_bin, y_train)
    score_lr = f1_score(y_test, modelo_lr.predict(X_test_bin), average='macro')
    
    modelo_nb = MultinomialNB()
    modelo_nb.fit(X_train_bin, y_train)
    score_nb = f1_score(y_test, modelo_nb.predict(X_test_bin), average='macro')
    
    modelo_svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=5000)
    modelo_svm.fit(X_train_bin, y_train)
    score_svm = f1_score(y_test, modelo_svm.predict(X_test_bin), average='macro')
    
    resultados.append({
        'Preprocesamiento': col, 
        'Macro-F1 (Reg. Logística)': score_lr,
        'Macro-F1 (Naive Bayes)': score_nb,
        'Macro-F1 (SVM)': score_svm
    })

df_resultados = pd.DataFrame(resultados).sort_values(by='Macro-F1 (Reg. Logística)', ascending=False)

print(df_resultados.to_string(index=False))

Preprocesamiento  Macro-F1 (Reg. Logística)  Macro-F1 (Naive Bayes)  Macro-F1 (SVM)
  text_lema_nltk                   0.569591                0.501426        0.520401
  text_stem_nltk                   0.568654                0.508922        0.525468
       text_lema                   0.568181                0.505213        0.537740
     text_basico                   0.553169                0.501394        0.526847
        text_pos                   0.536594                0.515326        0.497038
